# PrognosticAI on Kaggle: GPU training with MLflow + Dagster

Runs the repo's test suite, then the full Dagster pipeline (data download, data-quality check,
XGBoost training on GPU, RMSE gate) with every run tracked in MLflow.

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100), and Settings → **Internet on**
(needs a phone-verified Kaggle account). Then *Run All*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = "https://github.com/BinuShefieldShifani/PrognosticAI.git"
BRANCH = "main"   # change to test a feature branch before merging

%cd /kaggle/working
!rm -rf PrognosticAI && git clone -q --branch {BRANCH} {REPO}
%cd /kaggle/working/PrognosticAI
!git log --oneline -1

In [ ]:
# xgboost, pandas and scikit-learn are preinstalled on Kaggle
!pip install -q "mlflow>=3.0,<4" "dagster>=1.10,<2" pytest

In [ ]:
import os, sys
os.environ["PROGNOSTIC_DEVICE"] = "cuda"
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow.db"
sys.path.insert(0, os.getcwd())

!python download_data.py

## 1. Test suite (synthetic data, CPU, about 30 s)

In [ ]:
!python -m pytest tests/test_data_processor.py tests/test_model.py tests/test_training_mlflow.py tests/test_pipeline.py

## 2. Full pipeline on GPU via Dagster

In [ ]:
from dagster import materialize
from pipelines.definitions import cmapss_raw_files, rul_model, raw_data_is_valid, rmse_within_gate

result = materialize(
    [cmapss_raw_files, rul_model, raw_data_is_valid, rmse_within_gate],
    run_config={"ops": {"rul_model": {"config": {
        "device": "cuda",
        "register_model": True,
        "rmse_gate": 15.0,
    }}}},
)
print("Run success:", result.success)
for check in result.get_asset_check_evaluations():
    print(f"  check {check.check_name}: {'PASSED' if check.passed else 'FAILED'}")
metrics = result.output_for_node("rul_model")
{k: round(v, 3) if isinstance(v, float) else v for k, v in metrics.items()}

## 3. Compare runs in MLflow

In [ ]:
import mlflow
import matplotlib.pyplot as plt

mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs(experiment_names=["prognosticai-fd001"], order_by=["start_time DESC"])
display(runs[["tags.mlflow.runName", "tags.runner", "params.device",
              "metrics.val_rmse", "metrics.test_rmse", "metrics.nasa_score",
              "metrics.best_iteration", "metrics.train_seconds"]])

history = mlflow.MlflowClient().get_metric_history(metrics["mlflow_run_id"], "val_rmse_curve")
plt.plot([m.step for m in history], [m.value for m in history])
plt.xlabel("boosting round"); plt.ylabel("validation RMSE (cycles)")
plt.title("Validation curve (held-out engines)"); plt.show()

## 4. Try a variation (optional)
Change a parameter and re-run to get a second MLflow run to compare against, for example a lower learning rate.

In [ ]:
from model_trainer import train
variant = train(learning_rate=0.01, device="cuda", run_name="kaggle-lr0.01")
{k: round(v, 3) if isinstance(v, float) else v for k, v in variant.items()}

## 5. Download the tracking data
Download `prognosticai_mlflow.zip` from the Output panel. Locally: unzip, then
`mlflow ui --backend-store-uri sqlite:///mlflow.db` to browse params and metrics.
(Artifact paths point at /kaggle/working, so artifacts open only inside Kaggle.)

In [ ]:
!zip -qr /kaggle/working/prognosticai_mlflow.zip mlflow.db mlruns
!ls -lh /kaggle/working/prognosticai_mlflow.zip